## 3. Custom Feature Engineering & Preprocessing Pipeline

In [ ]:
from feature_selection_analysis import HouseFeatureEngineer

print("HouseFeatureEngineer imported from feature_selection_analysis.py.")

In [ ]:
from feature_selection_analysis import high_correlation_pairs

engineered_dev = HouseFeatureEngineer().fit_transform(X_dev)
redundant_pairs = high_correlation_pairs(engineered_dev, threshold=0.85)
print('Highly correlated numeric feature pairs (|r| >= 0.85, development split only):')
display(redundant_pairs)

## 4. Exploratory Data Diagnostics (Mutual Information on Development Partition)

In [ ]:
X_mi = X_dev.copy()
discrete_mask = []
for colname in X_mi.columns:
    if colname == 'MSSubClass' or X_mi[colname].dtype == 'object':
        X_mi[colname], _ = X_mi[colname].factorize()
        discrete_mask.append(True)
    else:
        discrete_mask.append(False)

X_mi = X_mi.fillna(0)
mi_scores = mutual_info_regression(X_mi, y_dev, discrete_features=discrete_mask, random_state=42)
mi_series = pd.Series(mi_scores, index=X_mi.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=mi_series.values[:20], y=mi_series.index[:20], hue=mi_series.index[:20], palette='viridis', legend=False)
plt.title("Top 20 Features by Mutual Information (Development Split Only)")
plt.xlabel("Mutual Information Score")
plt.show()

## 6. Fair Feature-Selection Comparison

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE


def custom_mi_scorer(X, y):
    if hasattr(X, 'toarray'):
        X = X.toarray()
    return mutual_info_regression(
        X, y, discrete_features=False, random_state=42
    )


selection_mi = SelectPercentile(score_func=custom_mi_scorer, percentile=80)
selection_rfe = RFE(
    estimator=RandomForestRegressor(
        n_estimators=50, min_samples_leaf=2, random_state=42, n_jobs=1
    ),
    n_features_to_select=0.8,
    step=0.2
)
base_model = models[screening_winner_name]

pipe_raw = Pipeline(steps=[
    ('preprocessor', clone(preprocessor)),
    ('model', clone(base_model))
])
pipe_passthrough = Pipeline(steps=[
    ('engineering', HouseFeatureEngineer()),
    ('preprocessor', clone(preprocessor)),
    ('model', clone(base_model))
])
pipe_mi = Pipeline(steps=[
    ('engineering', HouseFeatureEngineer()),
    ('preprocessor', clone(preprocessor)),
    ('selection', clone(selection_mi)),
    ('model', clone(base_model))
])
pipe_lasso = Pipeline(steps=[
    ('engineering', HouseFeatureEngineer()),
    ('preprocessor', clone(preprocessor)),
    ('selection', clone(selection_step)),
    ('model', clone(base_model))
])
pipe_rfe = Pipeline(steps=[
    ('engineering', HouseFeatureEngineer()),
    ('preprocessor', clone(preprocessor)),
    ('selection', clone(selection_rfe)),
    ('model', clone(base_model))
])

feature_selection_results = {}


def evaluate_feature_selector(name, estimator):
    scores = cross_validate(
        clone(estimator), X_dev, y_dev, cv=cv, scoring=scoring,
        n_jobs=-1, error_score='raise'
    )
    feature_selection_results[name] = {
        'Feature Set / Selection': name,
        'CV MAE Mean': -scores['test_MAE'].mean(),
        'CV MAE Std': scores['test_MAE'].std(),
        'CV MSE Mean': -scores['test_MSE'].mean(),
        'CV RMSE Mean': -scores['test_RMSE'].mean(),
        'CV R2 Mean': scores['test_R2'].mean()
    }
    print(
        f"Evaluated: {name} | CV MAE: "
        f"${feature_selection_results[name]['CV MAE Mean']:,.2f} "
        f"(Std: ${feature_selection_results[name]['CV MAE Std']:,.2f})"
    )


print("=== RAW, ENGINEERED, AND SELECTED FEATURE COMPARISON ===")
evaluate_feature_selector("Raw Features (Baseline)", pipe_raw)
evaluate_feature_selector("Engineered Features (All)", pipe_passthrough)
evaluate_feature_selector("Engineered + Mutual Information", pipe_mi)
evaluate_feature_selector("Engineered + Lasso", pipe_lasso)
evaluate_feature_selector("Engineered + RFE", pipe_rfe)

selector_choices = {
    "Engineered Features (All)": 'passthrough',
    "Engineered + Mutual Information": selection_mi,
    "Engineered + Lasso": selection_step,
    "Engineered + RFE": selection_rfe
}
chosen_selector_name = min(
    selector_choices,
    key=lambda name: feature_selection_results[name]['CV MAE Mean']
)
chosen_selector = selector_choices[chosen_selector_name]
print(f"Selected engineered feature strategy: {chosen_selector_name}")

selected_steps = [
    ('engineering', HouseFeatureEngineer()),
    ('preprocessor', clone(preprocessor))
]
if chosen_selector != 'passthrough':
    selected_steps.append(('selection', clone(chosen_selector)))
selected_steps.append(('model', clone(base_model)))
selected_dev_model = Pipeline(steps=selected_steps).fit(X_dev, y_dev)

feature_names = selected_dev_model.named_steps['preprocessor'].get_feature_names_out()
fitted_selector = selected_dev_model.named_steps.get('selection', 'passthrough')
if fitted_selector != 'passthrough':
    feature_names = feature_names[fitted_selector.get_support()]
selected_features_df = pd.DataFrame({'Selected Feature': feature_names})
print(f"Selected feature count: {len(selected_features_df)}")
display(selected_features_df)

pip_fit = Pipeline(steps=[
    ('engineering', HouseFeatureEngineer()),
    ('preprocessor', clone(preprocessor))
])
X_preprocessed = pip_fit.fit_transform(X_dev, y_dev)
all_features = pip_fit.named_steps['preprocessor'].get_feature_names_out()

lasso_fit = Lasso(alpha=100, max_iter=10000, tol=0.01, random_state=42)
lasso_fit.fit(X_preprocessed, y_dev)
coef_df = pd.DataFrame({
    'Feature': [feature.split('__')[-1] for feature in all_features],
    'Coefficient': lasso_fit.coef_,
    'AbsCoef': np.abs(lasso_fit.coef_)
}).sort_values(by='AbsCoef', ascending=False).head(30)
coef_df = coef_df[coef_df['AbsCoef'] > 0]

plt.figure(figsize=(10, 8))
sns.barplot(x='AbsCoef', y='Feature', data=coef_df, palette='magma', hue='Feature', legend=False)
plt.title("Top Predictive Features Selected by Lasso (Absolute Coefficients)")
plt.xlabel("Lasso Coefficient Weight Magnitude")
plt.ylabel("Feature Name")
plt.show()

In [ ]:
# Dedicated Feature Selection Comparison Evidence Table

fs_results_df = pd.DataFrame(feature_selection_results.values())
print("=== FEATURE SELECTION METHODS COMPARISON (5-Fold CV on X_dev) ===")
display(fs_results_df.sort_values(by='CV MAE Mean'))

### Selection Evidence and Decision Rule

The comparison table above reports cross-validated MAE (mean and standard deviation), MSE, RMSE, and $R^2$ for the raw-feature baseline, the complete engineered feature set, and each engineered selection strategy. The winning selector is chosen from the engineered candidates using the lowest mean validation MAE, then refit on `X_dev`; the selected-feature table above records the resulting feature subset. All preprocessing and selection steps run inside each CV fold to avoid fitting selectors on validation folds.

Mutual information screens transformed features individually, Lasso selects using a sparse linear model, and RFE recursively removes features using a random-forest importance estimator. The correlation report flags numeric feature pairs with absolute Pearson correlation of at least 0.85 for review; it is diagnostic and does not automatically discard features. Results are generated from the current data and should be rerun rather than copied as fixed scores.

## 12. Model Interpretability & Feature Weights

In [ ]:
fitted_preprocessor = selected_dev_model.named_steps['preprocessor']
fitted_selector = selected_dev_model.named_steps.get('selection', 'passthrough')
fitted_model = selected_dev_model.named_steps['model']

if isinstance(fitted_model, TransformedTargetRegressor):
    actual_model = fitted_model.regressor_
else:
    actual_model = fitted_model

feature_names = fitted_preprocessor.get_feature_names_out()
if fitted_selector != 'passthrough':
    feature_names = feature_names[fitted_selector.get_support()]

if hasattr(actual_model, 'feature_importances_'):
    importances = actual_model.feature_importances_
    assert len(feature_names) == len(importances)
    feat_imp_df = pd.DataFrame({'Feature': feature_names, 'Weight': importances}).sort_values(by='Weight', ascending=False).head(20)
    title = 'Top 20 Most Influential Features (Feature Importances)'
    xlabel = 'Importance Score'
elif hasattr(actual_model, 'coef_'):
    coefs = actual_model.coef_
    assert len(feature_names) == len(coefs)
    feat_imp_df = pd.DataFrame({'Feature': feature_names, 'Weight': np.abs(coefs)}).sort_values(by='Weight', ascending=False).head(20)
    title = 'Top 20 Most Influential Features (Lasso Coefficients Magnitude)'
    xlabel = 'Absolute Coefficient Magnitude'
else:
    feat_imp_df = None

if feat_imp_df is not None:
    feat_imp_df['Feature'] = feat_imp_df['Feature'].str.split('__').str[-1]
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Weight', y='Feature', data=feat_imp_df, palette='magma', hue='Feature', legend=False)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel('Feature Name')
    plt.show()
else:
    print("The winning model architecture does not support raw weights visualization.")